# WarehouseSort — Starter Notebook

This notebook walks through the **state IL (main track)** end-to-end on the easy level:
install → look at the env → generate demos → train a Diffusion Policy → evaluate.

**Requirements:** a CUDA GPU (ManiSkill 3 runs on GPU). In Google Colab, go to
*Runtime → Change runtime type → T4 GPU*.

References:
- [ManiSkill 3](https://maniskill.ai) — GPU-accelerated robot simulation
- [Diffusion Policy](https://diffusion-policy.cs.columbia.edu) — Chi et al. 2023

## 1. Install

In [ ]:
# Install ManiSkill and dependencies (takes ~2 min on Colab)
!pip install mani-skill==3.0.1 diffusers==0.27.2 gymnasium torch torchvision -q

# Clone the repo (skip if already in the repo directory)
import os
if not os.path.exists('warehouse_sort'):
    !git clone <YOUR_REPO_URL>
    %cd marso_hackathon

# Install the package
!pip install -e . -q

# Colab headless rendering (offscreen Vulkan)
import os
os.environ['DISPLAY'] = ''
os.environ['PYOPENGL_PLATFORM'] = 'egl'

## 2. Look at the environment

**Easy level**: 2 parcels (1 red-tagged, 1 blue-tagged), 2 color-coded bins (left/right),
fixed positions. State observation: parcel poses + tag colors + bin positions + proprioception.

In [ ]:
import gymnasium as gym
import torch
import warehouse_sort  # registers WarehouseSort-v1

env = gym.make(
    'WarehouseSort-v1',
    num_envs=1,
    obs_mode='state',
    control_mode='pd_ee_delta_pos',
    sim_backend='gpu',
    render_mode='rgb_array',
    difficulty='easy',
    num_parcels=2,
    fixed_poses=True,
)

obs, _ = env.reset(seed=42)
print('Observation shape:', obs.shape)   # (1, 54) for easy
print('Action space:     ', env.single_action_space)  # Box(-1,1,(4,))
print()
print('State vector layout (easy, 2 parcels):')
print('  [0:9]   qpos (joint positions)')
print('  [9:18]  qvel (joint velocities)')
print('  [18:25] tcp_pose (xyz + quaternion)')
print('  [25:26] is_grasped')
print('  [26:40] parcel poses (2 × 7)')
print('  [40:44] parcel tag colors one-hot (2 × 2: [red, blue])')
print('  [44:50] bin positions (xyz of red bin, then blue bin)')
print('  [50:54] bin color one-hot (identity matrix)')

In [ ]:
# Render the scene
from IPython.display import Image as IPImage
import PIL.Image, io

frame = env.render()                    # (1, H, W, 3) uint8
img = PIL.Image.fromarray(frame[0].cpu().numpy())
buf = io.BytesIO()
img.save(buf, format='PNG')
IPImage(buf.getvalue())

In [ ]:
# Run the scripted waypoint policy to verify the env works
import sys
sys.path.insert(0, '.')
from examples.scripted_policy import scripted_episode

env.close()
env = gym.make('WarehouseSort-v1', num_envs=1, obs_mode='state',
               control_mode='pd_ee_delta_pos', sim_backend='gpu',
               difficulty='easy', num_parcels=2, fixed_poses=True, max_episode_steps=200)

history = scripted_episode(env, max_steps=200, seed=42)
final_info = history[-1][-1]
print(f"Sorted: {final_info['success_count'].item():.0f} / 2")
env.close()

## 3. Generate demonstrations

Records 60 episodes of the scripted policy with a little action noise to spread the
state distribution (standard behavior cloning trick). Takes ~5 min on a T4.

In [ ]:
!python il/gen_demos.py --num-episodes 60 --action-noise 0.05 --difficulty easy

In [ ]:
# Check what was generated
import glob
files = glob.glob('il/demos/easy/*.h5')
for f in sorted(files):
    print(f)

## 4. Train a state Diffusion Policy

[Diffusion Policy](https://diffusion-policy.cs.columbia.edu) (Chi et al. 2023) — a plain MLP
behavior cloner fails on this task due to compounding error; DP's action chunking fixes it.

This uses a short run (`total_iters=10000`) to verify the pipeline. **For real training scale
up to `total_iters=30000` or more.** A full training run takes ~30–60 min on a T4.

In [ ]:
# Quick training run (verify pipeline; not fully converged)
!python il/train.py method=dp demo_dir=easy \
    flags.total_iters=10000 \
    flags.eval_freq=5000 \
    flags.exp_name=warehouse_state_dp_starter

In [ ]:
# For real training (uncomment and run — ~30-60 min)
# !python il/train.py method=dp demo_dir=easy \
#     flags.total_iters=30000 \
#     flags.eval_freq=5000 \
#     flags.exp_name=warehouse_state_dp_easy

## 5. Evaluate the checkpoint

In [ ]:
import glob, os

# Find the latest checkpoint
ckpts = sorted(glob.glob(
    'il/baselines/diffusion_policy/runs/warehouse_state_dp_starter/checkpoints/*.pt'
))
if not ckpts:
    print('No checkpoint found — run training first')
else:
    ckpt = ckpts[-1]
    print(f'Using checkpoint: {ckpt}')
    !python eval.py difficulty=easy obs_mode=state \
        policy=warehouse_sort.il_policy:load_dp \
        checkpoint={ckpt} \
        eval_config=conf/eval/default.yaml

## 6. Scaling to medium and hard

The same pipeline works for medium (4 parcels) and hard (6 parcels). You just need more
demos and more training iterations — the env, policy entrypoint, and eval command are identical.

```bash
# Medium (4 parcels, randomized positions)
python il/gen_demos.py --difficulty medium --num-episodes 100 --action-noise 0.05
python il/train.py method=dp demo_dir=medium flags.total_iters=50000 flags.exp_name=dp_medium
python eval.py difficulty=medium obs_mode=state \
    policy=warehouse_sort.il_policy:load_dp \
    checkpoint=il/baselines/diffusion_policy/runs/dp_medium/checkpoints/best_eval_success_at_end.pt \
    eval_config=conf/eval/default.yaml

# Hard (6 parcels, positions randomized + bins swap sides)
python il/gen_demos.py --difficulty hard --num-episodes 150 --action-noise 0.05
python il/train.py method=dp demo_dir=hard flags.total_iters=80000 flags.exp_name=dp_hard
python eval.py difficulty=hard obs_mode=state \
    policy=warehouse_sort.il_policy:load_dp \
    checkpoint=il/baselines/diffusion_policy/runs/dp_hard/checkpoints/best_eval_success_at_end.pt \
    eval_config=conf/eval/default.yaml
```

**Tips:**
- More demos help; `--action-noise` spreads the distribution.
- The state vector includes bin positions, so the policy can in principle read where the
  bins are each episode — key for hard where they swap.
- Judging uses held-out seeds with slightly wider position randomization; building a policy
  that generalizes across layouts (not just the training seeds) is the challenge.

## 7. Advanced track — image IL (template, not yet working)

An RGB Diffusion Policy template is provided based on the ManiSkill IL baselines and
[LeRobot](https://github.com/huggingface/lerobot) conventions. The architecture (ResNet18 +
SpatialSoftmax + ConditionalUNet1D) is wired up, but **image IL is not yet solving this task**.
It is an open problem for the advanced track.

```bash
# Generate RGB demos
python il/gen_demos.py --num-episodes 100 --action-noise 0.05  # also produces trajectory.rgb.*.h5

# Train RGB DP
python il/train.py method=dp_rgb demo_dir=easy

# Eval RGB DP
python eval.py difficulty=easy obs_mode=rgb obs_camera=scene \
    policy=warehouse_sort.il_policy:load_dp_rgb \
    checkpoint=<rgb_checkpoint> \
    eval_config=conf/eval/default.yaml
```